In [1]:
import os
os.environ['DISABLE_MODEL_SOURCE_CHECK'] = 'True'

import re
import json
import numpy as np
import cv2
from pdf2image import convert_from_path
from paddleocr import PaddleOCR
from docling.document_converter import DocumentConverter

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [2]:
# Initialisation de PaddleOCR, à faire une seule fois avant de lancer les scripts sur nos documents
ocr = PaddleOCR(use_angle_cls=True, lang='fr')
# Initialisation de DocumentConverter (idem)
converter = DocumentConverter()

/tmp/ipykernel_1673/3797138063.py:2: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='fr')
/home/oclaich/anaconda3/envs/myenv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/oclaich/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/oclaich/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LC

In [ ]:
def clean_amt(text):
    if not text: return 0.0
    # On retire les symboles monétaires et on gère la virgule
    clean = re.sub(r'[^\d,\.]', '', str(text)).replace(',', '.')
    try:
        return float(clean)
    except:
        return 0.0

In [ ]:
def process_document(pdf_path):
    """
    Fonction permettant de récupérer les informations contenues dans un document de manière structurée.
    On commence par extraire tout le contenu du pdf à l'aide des packages Docling ou OCR, puis on organise le rendu sous forme d'un dictionnaire json.
    Args:
        pdf_path (str): le chemin vers le fichier concerné
    Returns:
        final_json (json): dictionnaire json contenant les infos compartimentées relatives au document

    """
    filename = os.path.basename(pdf_path)
    print(f"--- Analyse du document : {filename} ---")
    
    # Tentative d'extraction des données à l'aide de Docling
    result = converter.convert(pdf_path)
    doc_text = result.document.export_to_markdown()
    
    # Si Docling ne renvoie presque rien, on passe à l'OCR
    if len(doc_text.strip()) < 50:
        print("Extraction des données avec OCR")
        images = convert_from_path(pdf_path, dpi=200)
        img_np = np.array(images[0])
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        
        ocr_res = ocr.ocr(img_bgr, cls=True)
        # On reconstruit les lignes
        lines = []
        if ocr_res[0]:
            for line in ocr_res[0]:
                lines.append(line[1][0])
        doc_text = "\n".join(lines)

    print(doc_text)

    final_json = structuration_finale(doc_text, filename)
    
    return final_json


def structuration_finale(raw_text, filename):
    """
    Fonction servant à organiser le contenu d'un document comptable sous forme d'un dictionnaire json.
    Args:
        raw_text (str): Contenu brut du document (extrait au préalable du pdf)
        filename (str): Nom du document
    Returns:
        res (json): Dictionnaire json contenant les informations importantes du document stockées de manière organisée
    """
    res = {
        "titre": f"{filename}",
        "type": "Inconnu",
        "date": None,
        "client": None,
        "fournisseur": None,
        "reference": None,
        "contenu": [],
        "totaux": None
    }

    lines = raw_text.split('\n')
    
    # Détection du type de document
    if any(x in raw_text.lower() for x in ['facture', 'invoice']):
        res["type"] = "facture"
    elif any(x in raw_text.lower() for x in ['relevé', 'compte']):
        res["type"] = "releve_de_compte"
    elif any(x in raw_text.lower() for x in ['ticket']):
        res["type"] = "note_de_frais"


    if res["type"] == "facture" :
        res["date"] = get_date(raw_text)
        res["client"] = get_client(raw_text)
        res["contenu"] = get_articles_lines(raw_text)
        res["totaux"] = get_total_values(raw_text)

    return res

def get_date(text):
    # Tenter de trouver la date liée à un mot précis
    anchor_match = re.search(r"(?:Facture\s+du|le|Date\s*:?)\s+([\d/\.-]{8,10})", text, re.IGNORECASE)
    if anchor_match:
        return anchor_match.group(1)

    # Si non trouvé, prendre toutes les dates du début du document
    lines = text.splitlines()[:15]
    header_text = "\n".join(lines)
    all_dates = re.findall(r"\d{2}[/\-]\d{2}[/\-]\d{2,4}", header_text)
    
    if all_dates:
        return all_dates[0]
        
    return None

def get_client(text):
    pattern = r'(?i)(?:code|réf\.?)\s*client\s*[:\-]?(?:\s+)?(\S+)'
    match = re.search(pattern, text)
    return match.group(1).strip() if match else None

def get_articles_lines(raw_text):
    lines = [l.strip() for l in raw_text.split('\n') if l.strip()]
    
    items = []
    col_map = {}
    header_found = False
    
    # Mots-clés pour identifier les colonnes (plusieurs variantes par champ)
    keywords = {
        'code': ['code', 'réf', 'article'],
        'libelle': ['libellé', 'désignation', 'description'],
        'quantite': ['quantité', 'qté', 'qte'],
        'pu': ['p.u', 'prix unit', 'prix u'],
        'montant': ['montant', 'total ht', 'prix net']
    }

    for line in lines:
        # Nettoyage des pipes et séparation en cellules
        cells = [c.strip() for c in line.split('|') if c.strip()]
        if not cells: continue

        # Recherche du header
        if not header_found:
            # On vérifie si cette ligne contient au moins 2 mots-clés de colonnes
            matches = 0
            temp_map = {}
            for i, cell in enumerate(cells):
                for field, synonyms in keywords.items():
                    if any(syn in cell.lower() for syn in synonyms):
                        temp_map[field] = i
                        matches += 1
            
            if matches >= 3: # On a trouvé notre en-tête
                col_map = temp_map
                header_found = True
            continue

        # Extraction des articles une fois le header connu
        # On vérifie si la ligne contient des données (un prix ou un code)
        # et qu'elle n'est pas une ligne de séparation (---)
        if header_found and not all(c == '-' for c in cells[0]):
            # Sécurité : vérifier qu'il y a assez de colonnes et un chiffre à la fin
            if len(cells) >= len(col_map) and re.search(r'\d', cells[-1]):
                item = {field: cells[idx] if idx < len(cells) else "" 
                        for field, idx in col_map.items()}
                items.append(item)
                
    return items

def get_total_values(raw_text):

    resultats = {}
    
    # Dictionnaire de configuration : Nom du champ -> Liste des variantes de mots-clés
    cibles = {
        'montant_ht': [r'total\s+ht', r'ht\s+hors\s+taxe', r'montant\s+ht'],
        'montant_tva': [r'tva', r'montant\s+t\.?v\.?a', r'taxe\s+sur\s+la\s+valeur\s+ajoutée'],
        'montant_ttc': [r'total\s+ttc', r'montant\s+ttc', r'total\s+toutes\s+taxes\s+comprises'],
        'a_payer': [r'à\s+payer', r'net\s+à\s+payer', r'total\s+dû', r'montant\s+total\s+dû']
    }
    
    # Regex de base pour un montant financier
    regex_nombre = r'(\d[\d\s.,]*[.,]\d{2})'

    for champ, variantes in cibles.items():
        for variante in variantes:
            # On cherche : le mot-clé + éventuellement des caractères/espaces + le montant
            pattern = rf'(?i){variante}.*?{regex_nombre}'
            match = re.search(pattern, raw_text, re.DOTALL)
            
            if match:
                valeur_brute = match.group(1)
                # Nettoyage pour conversion numérique (ex: "1 250,50" -> 1250.50)
                valeur_propre = valeur_brute.replace(' ', '').replace(',', '.')
                resultats[champ] = valeur_propre
                break # On s'arrête au premier match trouvé pour ce champ
                
    return resultats
    

In [ ]:
def clean_and_split_numeric(text, expected_count):
    """Extrait les nombres financiers d'une cellule."""
    pattern = r'-?[\d.]+,[\d]{2}-?'
    matches = re.findall(pattern, text)
    
    # Si on a trop de matches (ex: un total cumulé à la fin), on prend les N premiers
    if len(matches) > expected_count:
        return matches[:expected_count]
    # S'il en manque, on complète avec du vide
    while len(matches) < expected_count:
        matches.append("")
    return matches

def split_libelle(text, expected_count):
    """Nettoie le libellé et le divise en N parties."""
    # Nettoyage du bruit courant (Mentions avant et après les produits)
    # On enlève tout ce qui précède les premiers mots en MAJUSCULES (les produits)
    # et tout ce qui suit les mentions légales comme "En cas de..."
    text = re.sub(r'.*?([A-Z]{3,})', r'\1', text) # Enlève le début jusqu'au 1er mot MAJ
    text = re.split(r'(?i)en\s+cas\s+de|garantie', text)[0].strip()
    
    # On cherche des doubles espaces ou des changements brusques de produits
    parts = re.split(r'\s{2,}', text)
    
    if len(parts) < expected_count:
        # Si pas de doubles espaces, on tente de couper par blocs de mots
        words = text.split()
        n = len(words) // expected_count
        return [" ".join(words[i*n:(i+1)*n]) for i in range(expected_count)]
    
    return parts[:expected_count]

def process_complex_line(line):
    # Séparation des colonnes par le pipe
    cells = [c.strip() for c in line.split('|') if c.strip()]
    
    # On définit le nombre d'articles basé sur la colonne Montant (avant-dernière)
    # Valeurs observées : 1.153,95 5.548,95 297,99 (3 montants dans l'exemple principal)
    col_montant = cells[7]
    montants = re.findall(r'[\d.]+,[\d]{2}', col_montant)
    nb_articles = len(montants)
    
    if nb_articles == 0: return "Aucun article détecté"

    # 2. Préparation des données par colonne
    # On applique un traitement adapté à la nature de la colonne
    data_map = {
        "libelle": split_libelle(cells[0], nb_articles),
        "poids": clean_and_split_numeric(cells[2], nb_articles),
        "unite": cells[3].split(), # "KG KG KG" -> ["KG", "KG", "KG"]
        "pu": clean_and_split_numeric(cells[4], nb_articles),
        "decote": clean_and_split_numeric(cells[5], nb_articles),
        "net_ht": clean_and_split_numeric(cells[6], nb_articles),
        "montant": montants,
        "tva": cells[8].split()
    }

    # Reconstruction des lignes finales
    final_lines = []
    for i in range(nb_articles):
        row = (
            f"Ligne {i+1} : {data_map['libelle'][i]} / {data_map['poids'][i]} / "
            f"{data_map['unite'][i] if i < len(data_map['unite']) else ''} / "
            f"{data_map['pu'][i]} / {data_map['decote'][i]} / "
            f"{data_map['net_ht'][i]} / {data_map['montant'][i]} / "
            f"{data_map['tva'][i] if i < len(data_map['tva']) else ''}"
        )
        final_lines.append(row)
    
    return final_lines

# Exécution test
ligne = "| COMMANDE n°0662900 Bon de livraison 3623098 du N° cmde 12574499 MF 157 19 99 FARINE T5 25KG FAR COLOMBE T5 P90 FBA JULIE T45 FA0 En cas de retard de paiement, | 02/06/20 M02PO                              | 3.925,00 18.810,00 990,00 23.725,00      | KG KG KG                     | 450,00 450,00 520,00         | 156,00- 155,00- 219,00-                      | 294,00 295,00 301,00           | 1.153,95 5.548,95 297,99 | 3 3 3   |"

resultats = process_complex_line(ligne)
for r in resultats:
    print(r)

Ligne 1 : COMMANDEFARINEFARCOLOMBEFBAJULIE / 3.925,00 / KG / 450,00 / 156,00- / 294,00 / 1.153,95 / 3
Ligne 2 : T45 / 18.810,00 / KG / 450,00 / 155,00- / 295,00 / 5.548,95 / 3
Ligne 3 : FA0 / 990,00 / KG / 520,00 / 219,00- / 301,00 / 297,99 / 3


In [ ]:
file = "facture3"
mon_pdf = f"../data/{file}.pdf"

if os.path.exists(mon_pdf):
    resultat_json = process_document(mon_pdf)
    
    # Affichage propre
    print("\n--- CONTENU STRUCTURÉ ---")
    print(json.dumps(resultat_json, indent=4, ensure_ascii=False))
    
    # Sauvegarde
    with open(f'extracted_data/structured_data_{file}.json', 'w', encoding='utf-8') as f:
        json.dump(resultat_json, f, ensure_ascii=False, indent=4)
else:
    print("Fichier .pdf non trouvé, indiquer un chemin valable.")

--- Analyse du document : facture3.pdf ---


[INFO] 2026-01-27 18:19:49,976 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-27 18:19:49,997 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-01-27 18:19:50,157 [RapidOCR] download_file.py:60: File exists and is valid: /home/oclaich/anaconda3/envs/myenv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-27 18:19:50,158 [RapidOCR] main.py:50: Using /home/oclaich/anaconda3/envs/myenv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-27 18:19:50,933 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-27 18:19:50,935 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-01-27 18:19:50,941 [RapidOCR] download_file.py:60: File exists and is valid: /home/oclaich/anaconda3/envs/myenv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-01-27 18:19:50,948 [RapidOCR] main.py:50: Using /home/oclaich/anaconda3/envs/myenv/lib/python3.12/site-pa

| Référence                           | Référence                            |
|-------------------------------------|--------------------------------------|
| N° Facture / Invoice N° 19158399 RI | Date Facture / Invoice Date 11/06/20 |
| N° Client / Customer N° 90210021    | Site livraison / Ex Warehouse M02PO  |

Voir nos conditions générales de vente au verso.

## Facturé à :

CLIENT 14 AVENUE VALLIER 82032 MONTAUBAN

## FACTURE

Invoice

<!-- image -->

| Quantité Quantity                                                                                                                                              | Libellé Marchandises Description of goods   | Poids Weight                             | U N                          | Prix unitaire Unit price     | Décote                                       | Prix net HT Price net of tax   | MONTANT Amount           | T V A   |
|----------------------------------------------------------------------------------------------------------